# The compile-cache seed of submission 18's server, build `5d76d788f1`

**What it makes:** the compile caches of submission 18's Flash-Next server, packed into `/kaggle/working/jit_seed`:
SGLang's JIT kernels, Triton's and inductor's kernels, TileLang's QSA kernels (now with the INT8 + a per-key scale index-key decode
kernel), and the CPU all-reduce's extension. Submission 18 attaches this output and boots its server once, with no warm
boot and nothing compiled.

**How:** the same server cells as submission 18, then the server's first launch in a fresh container, as submission 12
did it: a warm boot with the plain NCCL all-reduce compiles every kernel into `/tmp/cache`, the real server starts
with our CPU all-reduce, and the warm requests run against it. Then `launch_server.sh pack` writes the caches and the
fingerprint the reusing notebook checks: the GPUs, both compilers, Python, the venv, the patched SGLang tree, the
server's switches and command line (with `--model-path /tmp/model_qg`, the QG overlay: the submission builds the same
folder), and every file's size and hash.

**One notebook per seed build** (the user, 10-24). This one serves only build `5d76d788f1` (agi2/feat/sub18-levers), INT8 + a per-key scale index keys, X 7.5, mf 0.89, with submission 18's switches (section
5) at 48 slots. It is never run again once a submission uses it. A seed that does not match is refused by the reusing
notebook, which then boots with the warm boot: slower, never broken. Some kernels compile only at first use and may
not be reached by the warm requests: the exact top-k, the block tier's promote and plan kernels, and the stage cache's
kernels (the first round with host-resident groups). Those then compile mid-run in the reusing notebook, ~1-2 s each.

**Server:** 48 slots: RadixArk's NVFP4 Qwen3.8-Flash-Next with QG's re-rounded experts, our fine-tuned DFlash2 draft with its 32k candidate vocabulary, 384 code-aware experts a layer, the QSA block tier at 7.5x with the stage
cache and INT8 + a per-key scale index keys, the sparse verify accept, 52 GDN state slots, the vision encoder in groups of 4,096
tokens; `--mem-fraction-static` 0.89, decode CUDA graphs 1, 2, 4, 8, 12, 16, 24, 32, 40, 48, bf16 GDN state.

**Trajectory**
- 2026-10-30 03:55 IST: built (`docs/reports/1030_0326_sub18_build/build_sub18.py`) from submission 17's seed notebook
  (`1028-seed-flashnext-sub17-06e9a82-s24` v2) with submission 18's server (the user: "after 48x24k next we need to target 48x96k and make it work; we will first need seed then submission notebook").


## 1. Settings

The clock starts here. `SLOTS` and `EXACT_TURNS` are submission 18's: they set the server's CUDA graphs and
switches, which the seed's fingerprint covers.


In [ ]:
import glob, json, os, re, time

T0 = time.time()               # the clock starts here
SLOTS = 48                     # requests in flight on the Flash-Next server: its CUDA graphs and request rows
EXACT_TURNS = True             # the server ends every turn exactly, at its stop token or its budget

def check(ok, problem):        # a failed check stops the run
    if not ok:
        raise RuntimeError(problem)


## 2. The machine

Four L4s are required: the model is split across all four GPUs.


In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv
!echo "$(nproc) CPUs: $(grep -m1 'model name' /proc/cpuinfo | cut -d: -f2), avx512f: $(grep -c -m1 avx512f /proc/cpuinfo)" && free -g | head -2 && df -h /tmp /kaggle/working | tail -2

n_l4 = get_ipython().getoutput('nvidia-smi -L | grep -c "NVIDIA L4"')
n_l4 = int(n_l4[0]) if n_l4 and n_l4[0].isdigit() else 0
check(n_l4 == 4, f'expected 4 NVIDIA L4, found {n_l4}')


## 3. The offline runtime

`/kaggle/input` is a network mount, slow for one reader but fast for many. So all model files are read once, in parallel, in the background, while the runtime installs; the server then loads them from memory.

The runtime is the output of the notebook `1006-sglang-runtime-cp312`: a standalone Python 3.12 plus every wheel of the `lmsysorg/sglang:v0.5.21` image. It installs offline into a venv in `/tmp`, using `uv` from the same wheelhouse.

In [ ]:
%%bash
du -shL /kaggle/input/models
# 16 readers over the model files, timed, except the PLE files: the loader reads their rows straight into its
# pinned table (SGLANG_AGI2_PLE_DIRECT_READ, section 6), so caching them too would only evict other weights; nor
# RadixArk's 192 expert shards, which the QG overlay replaces (reading both would be ~203 GB on a ~188 GB host)
nohup setsid bash -c 'T=$(date +%s); find -L /kaggle/input/models -type f ! -name "*plefp8*" ! -path "*hf-7b719225242a*/layer-*-experts-*" -print0 | xargs -0 -P 16 -n 4 cat > /dev/null; echo "precache done in $(( $(date +%s) - T )) s"' > /tmp/precache.log 2>&1 &
echo "precaching in the background (pid $!)"
# The runtime's wheelhouse too: on 10-24 its mount (gcsfuse) read ~20 MB/s for one reader, and the install took 13 min
# until 16 parallel readers had pulled the 7 GB into the page cache (then 2 min)
RT=$(dirname "$(find /kaggle/input -maxdepth 4 -name python312.tar.gz.bin | head -1)")
nohup setsid bash -c "find -L $RT/wheels -type f -print0 | xargs -0 -P 16 -n 2 cat > /dev/null" > /tmp/wheel_precache.log 2>&1 &
echo "precaching the wheelhouse in the background (pid $!)"

In [ ]:
%%bash
set -e
RT=$(dirname "$(find /kaggle/input -maxdepth 4 -name python312.tar.gz.bin | head -1)")
echo "runtime: $RT"
SECONDS=0
tar -xzf $RT/python312.tar.gz.bin -C /tmp
/tmp/python/bin/python3.12 -m venv /tmp/sgl
/tmp/sgl/bin/pip install -q --no-index --find-links $RT/wheels uv
UV_LINK_MODE=copy /tmp/sgl/bin/uv pip install --python /tmp/sgl/bin/python --offline --no-index --no-deps \
    --find-links $RT/wheels -r $RT/requirements.lock
echo "installed in ${SECONDS} s"
/tmp/sgl/bin/python -c "import torch, sglang, flashinfer; print('torch', torch.__version__, '| sglang', sglang.__version__, '| flashinfer', flashinfer.__version__, '|', torch.cuda.device_count(), 'GPUs')"

## 4. Our SGLang build: Flash-Next on 4 × L4

One diff over the runtime's stock SGLang 0.5.21: `git diff v0.5.21 5d76d788f1 -- python/sglang` of our fork
(branch `agi2/feat/sub18-levers`, 1,029,632 bytes, 124 files, sha256 `6cf9769c02b0`). It is submission 17's build
(`agi2/feat/sub17` 06e9a8204c: the Flash-Next L4 stack, the QSA block tier with promote-on-touch and per-layer
residency, the exact top-k) plus these, each behind its own switch (section 5):
- **The sparse verify accept** (exact up to float rounding at the sampling CDF): DFlash's accept runs on the
  distributions it already has, the target's sharded top-20 and the draft selector's candidates, instead of scattering
  both into two full-vocabulary fp32 matrices. At 48 slots the sampler's reserve falls from 0.78 to 0.31 GB a GPU.
- **The stage cache** of the QSA block tier (exact: the attention reads the same bytes): each host-resident 4-token
  group a step selects is copied once into a GPU scratch and kept across steps until its rows change, so host RAM can
  hold most of the pool. X is 7.5 here: ~4.7M index slots a GPU.
- **INT8 + a per-key scale index keys**, the one lossy switch. The compressed keys that rank which 2,048 tokens each query attends are
  stored at ~56 % of their BF16 bytes and cast to BF16 on the GPU before the indexer's
  matmul; the attention itself reads the same K/V. On identical tokens (16 prompts of 16k), FP8 keys were at the
  run-to-run floor on the model's own continuations and 1.27x it on prompt text, with no significant shift.
  INT8 keys have ~5x less rounding error: on gemma-normed keys the top-512 overlap with BF16 is 99.3-99.9 % (FP8: 97.6-99.5 %), and their decode kernel matches an fp32 reference on the stored integers.
- **The draft's graph-index cap** (exact): the draft's FlashInfer CUDA-graph index buffer is sized for its 2,048-token
  window instead of 48 x 4 rows x the 262,144-token context. On Kaggle the draft graphs' memory fell from 0.62 to 0.22 GB,
  and 0.86 GB stays free at ready at 48 slots (0.40 without it).
- **The decode-lever bundle** (`agi2/feat/qsa-idx-shared` 17b4f71a70, merged in `5d76d788f1`): M-RoPE positions
  uploaded asynchronously, Marlin's `cache13` left empty, the expert-parallel top-k sum over local slots only, the EP id
  map as one Triton gather, and the draft's candidate head on a 32,768-token subset (the cell below). Per-kernel
  timing on one L4: −2.5 ms a step; on 4 × L40S +2.4 % end to end at unchanged acceptance.
- In the diff but off here: the page-granular host tier, the CPU all-reduce's prefill split, split-KV sparse decode,
  the top-k diagnostics, the grouped QSA indexer (BF16 keys only), the fused and int8 hyper-connection mix and the skinny GEMM.

Measured on Kaggle 4 × L4 at 48 slots: 48 x 24k at 140.6 ms a step with FP8 keys at X 6 (6.9 % host share, ~965 tok/s), against 179.7 ms for submission 17's server at 48 slots (`1030_0046`, `1029_2325`); 48 x 96k (FP8 keys, X 7.5, 4.74M index slots) at 173.3 ms a step with all 48 running, a median host share of 30 %, ~820 tok/s, 0 errors, no OOM or retraction (submission 17's 24-slot server: ~650-700 tok/s at long context).

Stored compressed (gzip + base64) to keep the notebook under Kaggle's 1 MiB source cap; the cell writes exactly the
diff's bytes. Then the 384-expert keep list (48 layers × 384 expert ids, the list order being the expert-parallel
placement) and the draft's candidate vocabulary, also stored compressed.


In [ ]:
# sglang-v0.5.21-to-sub18-5d76d78.diff: stock v0.5.21 -> agi2/feat/sub18-levers 5d76d788f1 (python/sglang), stored compressed
import base64, gzip, hashlib
data = gzip.decompress(base64.b64decode('''
...
'''))
open('sglang-v0.5.21-to-sub18-5d76d78.diff', 'wb').write(data)
print('sglang-v0.5.21-to-sub18-5d76d78.diff:', len(data), 'bytes, sha256', hashlib.sha256(data).hexdigest()[:12])
check(hashlib.sha256(data).hexdigest() == '6cf9769c02b0f677357f25b0bc8c483641ac518498e32ef009dcba0e3ad5401c', 'the build diff does not match its sha256')


In [ ]:
%%bash
set -e
SITE=$(/tmp/sgl/bin/python -c "import os, sglang; print(os.path.dirname(os.path.dirname(sglang.__file__)))")
patch -p2 --fuzz=0 --forward --no-backup-if-mismatch -d "$SITE" < sglang-v0.5.21-to-sub18-5d76d78.diff > /tmp/patch.log 2>&1 || { tail -20 /tmp/patch.log; exit 1; }
grep -c '^patching file' /tmp/patch.log
grep -q "def agi2_expert_keep_lists" "$SITE/sglang/srt/configs/qwen4_exp.py"
test -f "$SITE/sglang/srt/layers/attention/qsa/pair_layer_split.py"
test -f "$SITE/sglang/srt/speculative/dflash_length_cap.py"
test -f "$SITE/sglang/srt/models/qwen4_exp_ple_direct.py"
test -f "$SITE/sglang/srt/speculative/fn_conf.py"
test -f "$SITE/sglang/srt/mem_cache/agi2_qsa_block_tier.py"
test -f "$SITE/sglang/srt/mem_cache/agi2_qsa_host_tier.py"
test -f "$SITE/sglang/kernels/jit/csrc/elementwise/fast_topk_exact.cuh"
grep -q SGLANG_AGI2_FAST_TOPK_EXACT "$SITE/sglang/kernels/ops/attention/fast_topk.py"
grep -q SGLANG_AGI2_QSA_BLOCK_TIER_POT "$SITE/sglang/srt/mem_cache/agi2_qsa_block_tier.py"
grep -q SGLANG_AGI2_QSA_BLOCK_TIER_PERLAYER "$SITE/sglang/srt/mem_cache/agi2_qsa_block_tier.py"
grep -q "def stage_cache_enabled" "$SITE/sglang/srt/mem_cache/agi2_qsa_block_tier.py"
grep -q SGLANG_AGI2_QSA_INDEX_INT8 "$SITE/sglang/srt/mem_cache/qsa_kv_pool.py"
grep -q _tilelang_qsa_mqa_decode_int8k_kernel "$SITE/sglang/srt/layers/attention/qsa/mqa.py"
grep -q SGLANG_AGI2_SPARSE_ACCEPT "$SITE/sglang/srt/environ.py"
grep -q SGLANG_AGI2_SWA_GRAPH_IDX_CAP "$SITE/sglang/srt/layers/attention/flashinfer_backend.py"
grep -q SGLANG_AGI2_DRAFT_VOCAB "$SITE/sglang/srt/models/dflash.py"
grep -q SGLANG_AGI2_MOE_EP_SUM "$SITE/sglang/srt/layers/moe/fused_moe_triton/fused_marlin_moe.py"
echo "Flash-Next build in place: $SITE"


In [ ]:
# keep_384_code_bal4.json: 384 code-aware routed experts a layer (prose + code census), stored compressed
data = gzip.decompress(base64.b64decode('''
...
'''))
open('/tmp/keep_384_code_bal4.json', 'wb').write(data)
check(hashlib.sha256(data).hexdigest() == 'a4a75c4eb06b8a593b0c0a7374de17f31c2315d864db6090048ac685396dcd0b', 'the keep list does not match its sha256')
k = json.loads(data)
print('keep list:', len(k['layers']), 'layers x', sorted({len(v) for v in k['layers']}), 'experts, sha256', hashlib.sha256(data).hexdigest()[:12])


In [ ]:
# draft_vocab_32k.json: the DFlash2 draft's candidate head scores these token ids only (SGLANG_AGI2_DRAFT_VOCAB, the
# lever bundle; docs/reports/1029_0021_upstream_sweep_flashnext), stored compressed
import base64, gzip, hashlib
data = gzip.decompress(base64.b64decode('''
...
+5fe/qW3f+ntX3r7l95e//s/LNBpfIJVAwA=
'''))
open('/tmp/draft_vocab_32k.json', 'wb').write(data)
print('draft vocabulary:', len(json.loads(data)), 'token ids, sha256', hashlib.sha256(data).hexdigest()[:12])
check(hashlib.sha256(data).hexdigest() == 'f72ceca4e3822fa7fdfdab6c5989cc0142e4af7fdde197cc22fc083e492de098', 'the draft vocabulary does not match its sha256')


## 5. The SGLang server

Submission 17's server at 48 slots, in Kaggle's paths:
- **Model:** RadixArk's NVFP4 Qwen3.8-Flash-Next with its experts **re-rounded by GPTQ (QG)**: the same files, tensor
  names and layout, only the experts' `weight` and `weight_scale` re-solved against calibration activations (block error
  −63.5 %, KL to BF16 −25 %, acceptance x0.995; `docs/reports/1029_1845_nvfp4_rerounding_scope`). The launch builds an
  overlay folder, RadixArk's files with QG's 192 expert shards, and stops unless all 192 are in place.
- **Model split:** tensor parallel 4 × expert parallel 4. NVFP4 experts run as Marlin W4A16 (the L4 has no FP4 tensor
  cores); dense layers BF16; FP8 KV cache; bf16 GDN state; the embedding and the PLE n-gram table in pinned host RAM.
- **Experts:** the 384-expert keep list, in a census-balanced placement over the 4 GPUs.
- **KV pool:** the QSA pair layer split stores each QSA layer's KV on one GPU of its pair; the GDN ReplaySSM fold drops
  the speculative state snapshots. The QSA block tier (the build section above) at 7.5x with the stage cache and
  INT8 + a per-key scale index keys holds ~4.7M tokens a GPU (48 conversations of ~96k), cold 4-token groups waiting in host RAM.
- **Requests:** 48 decode at a time (`SLOTS`), decode CUDA graphs for 1, 2, 4, 8, 12, 16, 24, 32, 40, 48 (they must reach
  `SLOTS`; a batch between two sizes runs on the next one up); 50 request rows, because a conversation resting between
  tool calls keeps its row; 52 GDN state slots.
- **Streaming sessions** with retract, evict and park (an idle conversation waits in host RAM, up to 8 GB a GPU: the
  tier's host rows take ~13 GB a GPU at 7.5x).
- **Speculative decoding:** DFlash2, 4 draft tokens a step, with exact turn boundaries and the sparse verify accept.
- **All-reduce:** our int8 CPU reducer, 4 threads on the machine's last 4 cores; everything else stays off them.
- **Memory:** 89.0 % of each GPU. DFlash needs at least 1.0 GB free after the target's graphs, or it
  drops its own draft graph without an error and runs ~14 ms a step slower: the check after the launch reads it from
  the log. The vision encoder runs in groups of at most 4,096 visual tokens.

The launch is a file, `launch_server.sh`, so a crash mid-run restarts the server with the same switches
(`restart_server`). Its compile cache lives in `/tmp/cache`: SGLang's
JIT kernels, Triton, inductor, TileLang and the CPU all-reduce's extension.
- **With a seed attached** (this build's seed notebook): it unpacks the seed and checks everything the caches' keys
  depend on (`jit_seed_check.py`: the GPUs, both compilers, Python, the venv, the patched SGLang tree, the switches,
  the command line, the model path included, and every seeded file). If all match, it boots once.
- **With no seed, or one the check refuses:** it empties the cache and boots as submission 12 does. A warm boot with
  the plain NCCL all-reduce compiles every kernel, then the real server starts with our CPU all-reduce.
- **The weight loader** prefetches the checkpoint files except the PLE files and RadixArk's own expert shards, and loads
  with 2 threads. The PLE rows are read straight into their pinned tables.
- **The lm_head stays BF16** (submission 12 ran it in FP8): the one lossy piece with measured drift, for ~+4 % speed.



**Which features this server runs**, as environment variables the server inherits:


In [ ]:
SESSIONS = True   # streaming sessions hold the harness's conversations (section 9)
# The CPUs this container may use (cpu_count() also counts ones it may not: 48 against 46 on L4x4), kept from the
# first run, since the restriction below shrinks this kernel's own set.
CPUS = [int(c) for c in os.environ.setdefault('ARC_CPUS', ','.join(map(str, sorted(os.sched_getaffinity(0))))).split(',')]
SMR_THREADS = 4   # the reducer's threads (the tested config's 4) get the last cores to themselves
KEEP = CPUS[:-SMR_THREADS]
os.sched_setaffinity(0, KEEP)
KEEP_LIST = '/tmp/keep_384_code_bal4.json'   # written in section 5
# The final Modal round's server environment (config l4x4_flashnext_nvfp4_prune384code_16slots_kvpool_tuned_bf16head_
# loadopt_dflash2_mf90, round 1025_2021), with the block tier and the vision encode budget, and sub 17's switches
# (docs/reports/1026_1946_structural_decode/SUB17_SERVER.md)
FLAGS = dict(
    # exact pinned host buffers above 1 GB (the PLE tables and the session park; no 2x rounding)
    PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True,pinned_max_round_threshold_mb:1024',
    # the KV pool: GDN fold, QSA pair layer split, vision blocks in host RAM, the reserve cuts
    SGLANG_ENABLE_GDN_REPLAYSSM_FOLD='1', SGLANG_AGI2_QSA_PAIR_LAYER_SPLIT='1', SGLANG_AGI2_FN_VISION_HOST_BLOCKS='1',
    SGLANG_FLASHINFER_WORKSPACE_SIZE='100663296', SGLANG_VLM_CACHE_SIZE_MB='0',
    # the experts and the weights' layout
    SGLANG_AGI2_FN_EXPERT_KEEP=KEEP_LIST, SGLANG_AGI2_FN_EXPERT_KEEP_ANY_ORDER='1', SGLANG_AGI2_HOST_EMBED='1',
    SGLANG_AGI2_MARLIN_NO_SWIZZLED='1', SGLANG_AGI2_MARLIN_INPLACE_REPACK='1',
    # the PLE rows read straight into their pinned tables, and the loader's prefetch kept off the PLE files
    SGLANG_AGI2_PLE_DIRECT_READ='1', SGLANG_AGI2_PREFETCH_EXCLUDE='plefp8',
    # QSA decode on sm_89 and its speed levers
    SGLANG_AGI2_QSA_TRITON_DECODE='1', SGLANG_AGI2_QSA_IDX_GPC='16', SGLANG_AGI2_QSA_IDX_STAGES='1',
    SGLANG_AGI2_QSA_DIRECT_SPARSE='1',
    # the QSA block tier (sub 14), here at 7.5x for 48 slots (sub 18; 3.0x at 24 slots in subs 15-17): ~4.7M index
    # slots a GPU, 48 conversations of ~96k; 4-token groups unselected for a while go to host RAM
    # in rounds every 64 steps, keeping 4-9 % of the GPU rows free, at most 32,768 groups a round
    SGLANG_AGI2_QSA_BLOCK_TIER_X='7.5', SGLANG_AGI2_QSA_BLOCK_TIER_EVERY='64',
    SGLANG_AGI2_QSA_BLOCK_TIER_MAX_GROUPS='32768', SGLANG_AGI2_QSA_BLOCK_TIER_RESERVE='0.04',
    SGLANG_AGI2_QSA_BLOCK_TIER_LOW_WATER='0.06', SGLANG_AGI2_QSA_BLOCK_TIER_HYSTERESIS='0.03',
    # sub 17: promote-on-touch (host-resident groups a request selects move back to GPU rows in rounds; the audit
    # checks every round's books) with one residency class per QSA layer this GPU holds
    SGLANG_AGI2_QSA_BLOCK_TIER_POT='1', SGLANG_AGI2_QSA_BLOCK_TIER_POT_AUDIT='1',
    SGLANG_AGI2_QSA_BLOCK_TIER_PERLAYER='1',
    # sub 17: two host syncs dropped (QSA metadata, extend); fast_topk's overflow bucket refined exactly (upstream
    # clips a bucket of more than 4,096 candidates and drops true winners)
    SGLANG_AGI2_QSA_NOSYNC='1', SGLANG_AGI2_EXTEND_NOSYNC='1', SGLANG_AGI2_FAST_TOPK_EXACT='1',
    # sub 18: the stage cache (each host-resident group a step selects is copied once into a GPU scratch and kept
    # across steps: the same bytes, read on the GPU) behind a 2,048-token never-demote window
    SGLANG_AGI2_QSA_BLOCK_TIER_STAGE='1', SGLANG_AGI2_QSA_BLOCK_TIER_STAGE_MAX='16384',
    SGLANG_AGI2_QSA_BLOCK_TIER_STAGE_CACHE='1', SGLANG_AGI2_QSA_BLOCK_TIER_WINDOW='2048',
    # sub 18: the QSA index keys in INT8 + a per-key scale (the one lossy switch: they only rank which 2,048 tokens
    # each query attends; the attention reads the same K/V)
    SGLANG_AGI2_QSA_INDEX_INT8='1',
    # the vision encoder in groups of at most 4,096 visual tokens (its activations grow with the call; a 16-slot
    # Modal round ran out of GPU memory there, 1025_1906)
    SGLANG_MM_ENCODE_MAX_TOKENS='4096',
    # log every Triton compile after serving starts (the block tier's move kernel compiles its first few tail
    # sizes mid-run, ~1-2 s each; seed review 10-26). Not part of the seed's fingerprint (its ENV_SKIP)
    SGLANG_TRITON_SLOW_COMPILE_THRESHOLD_SECS='0',
    # DFlash2 and the verify path
    SGLANG_LOGPROB_CHUNK_SIZE='128', SGLANG_DFLASH_DRAFT_WINDOW_KV='1', SGLANG_AGI2_DRAFT_WINDOW_KV_PAGED='1',
    SGLANG_ENABLE_DFLASH_SHARDED_VERIFY='1', SGLANG_AGI2_VERIFY_TAIL='1', SGLANG_AGI2_EARLY_PUBLISH='1',
    SGLANG_AGI2_DFLASH_PINNED_NTNP='1',   # (the lm_head stays BF16: no SGLANG_ENABLE_FP8_LM_HEAD)
    SGLANG_AGI2_SWA_ONLY_WRAPPER0='1',
    # sub 18: the verify accept on the sparse distributions it already has (the sampler's reserve 0.78 -> 0.31 GB at
    # 48 slots) and the draft's graph-index buffer sized for its 2,048-token window (0.40 GB), both exact
    SGLANG_AGI2_SPARSE_ACCEPT='1', SGLANG_AGI2_SPARSE_ACCEPT_CHUNK='8', SGLANG_AGI2_SWA_GRAPH_IDX_CAP='1',
    # sub 18: the decode-lever bundle (agi2/feat/qsa-idx-shared 17b4f71a70): M-RoPE async upload, Marlin's
    # cache13 left empty, the EP top-k sum over local slots, the EP id map as one gather, and the draft's
    # candidate head on a 32,768-token subset (written in section 5)
    SGLANG_AGI2_MROPE_ASYNC='1', SGLANG_AGI2_MOE_EMPTY_CACHE13='1', SGLANG_AGI2_MOE_EP_SUM='1',
    SGLANG_AGI2_EP_MAP_TRITON='1', SGLANG_AGI2_DRAFT_VOCAB='/tmp/draft_vocab_32k.json',
    # the int8 CPU all-reduce, on the last SMR_THREADS cores
    SGLANG_ENABLE_SM_REDUCE_ALLREDUCE='1', SGLANG_SM_REDUCE_FMT='int8', SGLANG_SM_REDUCE_DOWNQ='1',
    SGLANG_SM_REDUCE_THREADS=str(SMR_THREADS), SGLANG_SM_REDUCE_CHUNK_KB='64', SGLANG_SM_REDUCE_PUBWARP='1',
    SGLANG_SM_REDUCE_CPUS=','.join(map(str, CPUS[-SMR_THREADS:])),
)
if SESSIONS:   # the session fixes: retract, evict, park; QSA's session prefix and two-group pending ring
    FLAGS.update(SGLANG_ENABLE_STREAMING_SESSION_RETRACT='1', SGLANG_ENABLE_STREAMING_SESSION_EVICT='1',
                 SGLANG_ENABLE_STREAMING_SESSION_PARK='1', SGLANG_STREAMING_SESSION_PARK_HOST_GB='8',
                 SGLANG_AGI2_QSA_SESSION_PREFIX='1', SGLANG_AGI2_QSA_RING2='1')
# Exact turn boundaries, only where their patch is in place: without it a cut turn is rebuilt, as in submission 8
EXACT_TURNS = EXACT_TURNS and bool(glob.glob('/tmp/sgl/lib/python3*/site-packages/sglang/srt/speculative/dflash_length_cap.py'))
if EXACT_TURNS:
    FLAGS.update(SGLANG_ENABLE_DFLASH_LENGTH_CAP='1', SGLANG_ENABLE_DFLASH_STOP_CAP='1',
                 SGLANG_ENABLE_STREAMING_SESSION_FOLDED_CLAIM='1')
FLAGS['ARC_SLOTS'] = str(SLOTS)                                 # read by the launch cell: CUDA graphs up to this batch
FLAGS['ARC_MAX_REQS'] = str(SLOTS + 2 if SESSIONS else SLOTS)   # resting sessions hold their rows: slots + 2
FLAGS['ARC_SESSIONS'] = '1' if SESSIONS else ''                 # read by the launch cell: --enable-streaming-session
# sub 14's server shape, read by the launch cell (unset there, they are sub 13's 0.92 / graphs to 12 / 18 slots)
FLAGS['ARC_MEM_FRACTION'] = '0.89'                              # --mem-fraction-static (sub 17: 0.896 at 24 slots)
FLAGS['ARC_GRAPH_BS'] = '1 2 4 8 12 16 24 32 40 48'             # --cuda-graph-bs-decode: 10 sizes up to SLOTS (sub 17:
                                                                #   1-24)
FLAGS['ARC_MAMBA_SLOTS'] = str(SLOTS + 4)                       # --max-mamba-cache-size: 52 GDN state slots
FLAGS['ARC_SSM_DTYPE'] = 'bfloat16'                             # --mamba-ssm-dtype: the GDN state in bf16 (sub 16: fp32)
for k in [k for k in os.environ if k.startswith(('SGLANG_', 'PYTORCH_CUDA_ALLOC', 'ARC_SLOTS', 'ARC_MAX_REQS', 'ARC_SESSIONS', 'ARC_MEM_FRACTION', 'ARC_GRAPH_BS', 'ARC_MAMBA_SLOTS', 'ARC_SSM_DTYPE'))]:
    os.environ.pop(k)  # a second server in the same session must not inherit the previous one's switches
os.environ.update(FLAGS)
check(os.path.exists(KEEP_LIST), f'the keep list {KEEP_LIST} is missing (section 5)')
print('exact turns' if EXACT_TURNS else 'no exact turns', FLAGS)
print(f'{len(CPUS)} CPUs; this kernel and the server on {len(KEEP)} of them, the reducer on {CPUS[-SMR_THREADS:]}')


In [ ]:
%%writefile launch_server.sh
# The server's launch, as a file: the cell below runs it, and so does restart_server() after a crash.
#   bash launch_server.sh            start the server (the first launch in a container also prepares the cache)
#   bash launch_server.sh pack DIR   write the compile cache of this container as a seed into DIR, and stop
# (Submission 12's launch_server.sh plus the seeded compile cache, docs/reports/1024_1137_jit_cache_seeding/, and the
#  weight loader's prefetch and threads, docs/reports/1024_1318_weight_loading_phase1.md)
[ -f sglang.log ] && [ -z "${1:-}" ] && mv sglang.log "sglang_$(date +%m%d_%H%M%S).log"   # keep the previous server's log (a crash's)
MODEL=$(dirname "$(find -L /kaggle/input/models -path '*qwen3-8-flash-next-nvfp4*' -name config.json | head -1)")
DRAFT=$(dirname "$(find -L /kaggle/input/models -path '*qwen3-8-flash-next-dflash2*' -name config.json | head -1)")
# sub 18: the QG experts (GPTQ-re-rounded NVFP4, docs/reports/1029_1845_nvfp4_rerounding_scope) as an overlay folder:
# RadixArk's files, with the 192 expert shards taken from the QG Model; fail closed unless all 192 are in place
QG=$(dirname "$(find -L /kaggle/input/models -path '*agi2-qg1-8352943a271b*' -name MANIFEST.sha256 | head -1)")
if [ "$QG" != "." ]; then
    rm -rf /tmp/model_qg && mkdir -p /tmp/model_qg
    for f in "$MODEL"/*; do
        b=$(basename "$f")
        case "$b" in
            layer-*-experts-*.safetensors) ln -s "$QG/$b" "/tmp/model_qg/$b" ;;
            *) ln -s "$f" "/tmp/model_qg/$b" ;;
        esac
    done
    [ "$(ls -L /tmp/model_qg/layer-*-experts-*.safetensors | wc -l)" = 192 ] || { echo "QG overlay incomplete"; exit 1; }
    MODEL=/tmp/model_qg
else
    echo "QG experts not attached"; exit 1
fi
echo "model: $MODEL"
echo "draft: $DRAFT"
mkdir -p /tmp/cache/{triton,inductor,flashinfer,xdg,torch_extensions,sglang_jit,sglang,cute_dsl}
# Kernels compiled at start-up must use the wheelhouse's CUDA 13 toolkit, not the image's CUDA 12.8. That toolkit
# pairs nvcc 13.4 with CUDA 13.0 runtime headers (as frozen in the lmsysorg image), so CCCL's exact-version check is off.
export CUDA_HOME=/tmp/sgl/lib/python3.12/site-packages/nvidia/cu13
# /tmp/sgl/bin comes last: the venv's ninja builds the JIT kernels if the image has no ninja of its own.
export PATH=$CUDA_HOME/bin:$PATH:/tmp/sgl/bin NVCC_APPEND_FLAGS="-DCCCL_DISABLE_CTK_COMPATIBILITY_CHECK"
export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:$CUDA_HOME/lib
ln -sfn lib $CUDA_HOME/lib64 && ln -sf libcudart.so.13 $CUDA_HOME/lib/libcudart.so  # the names the JIT linker asks for
env | grep -E '^(SGLANG_|PYTORCH_CUDA_ALLOC|ARC_(SLOTS|MAX_REQS|SESSIONS|MEM_FRACTION|GRAPH_BS|MAMBA_SLOTS|SSM_DTYPE))' | sort  # the switches set above
# ARC_MEM_FRACTION, ARC_GRAPH_BS, ARC_MAMBA_SLOTS (sub 14) and ARC_SSM_DTYPE (sub 17) come from the switches cell;
# unset, they are sub 13's values (0.92; decode graphs 1-8, 10, 12; 18 GDN state slots; fp32 GDN state), so earlier
# builds launch exactly as before. The GDN state dtype must be this flag: with --enable-linear-replayssm-spec the
# server overwrites the env var SGLANG_MAMBA_SSM_DTYPE with the flag's value (serving_hook.py:398-399).
export HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1 TOKENIZERS_PARALLELISM=false PYTHONUNBUFFERED=1
# Every compile cache in one real directory under /tmp (a cache path under a symlink never hits: seen on Modal).
# SGLANG_CACHE_DIR too: at import SGLang points TileLang's cache and CUDA's own PTX cache under it (environ.py
# third_party_cache_defaults); left unset they land in ~/.cache/sglang, outside the seeded tree.
export TRITON_CACHE_DIR=/tmp/cache/triton TORCHINDUCTOR_CACHE_DIR=/tmp/cache/inductor SGLANG_JIT_CACHE_DIR=/tmp/cache/sglang_jit
export FLASHINFER_WORKSPACE_BASE=/tmp/cache/flashinfer XDG_CACHE_HOME=/tmp/cache/xdg TORCH_EXTENSIONS_DIR=/tmp/cache/torch_extensions
export SGLANG_CACHE_DIR=/tmp/cache/sglang
# CuTe DSL (FlashInfer's rmsnorm kernels on sm_89) keeps its own file cache, by default /tmp/<user>/cutlass_python_cache;
# its key holds the working directory, so the server always starts from this one.
export CUTE_DSL_CACHE_DIR=/tmp/cache/cute_dsl
# The last line's two switches are the weight loader's (checkpoint prefetch, 2 loader threads): they change how the
# weights are read, not what is compiled, so the seed check leaves them out of the command line it compares.
ARGS=(--model-path "$MODEL" --speculative-draft-model-path "$DRAFT" --served-model-name qwen3.8-flash-next
      --host 127.0.0.1 --port 30000 --tp 4 --reasoning-parser qwen3 --context-length 262144 --mem-fraction-static ${ARC_MEM_FRACTION:-0.92}
      --max-running-requests $ARC_MAX_REQS --chunked-prefill-size 1024 --mamba-ssm-dtype ${ARC_SSM_DTYPE:-float32} --kv-cache-dtype fp8_e4m3
      --enable-metrics --decode-log-interval 200 --trust-remote-code --disable-radix-cache --ep-size 4
      --quantization modelopt_fp4 --fp4-gemm-backend marlin --moe-runner-backend marlin --page-size 64 --ple-offload-embedding
      --cuda-graph-max-bs-decode $ARC_SLOTS --cuda-graph-bs-decode ${ARC_GRAPH_BS:-1 2 3 4 5 6 7 8 10 12} --max-mamba-cache-size ${ARC_MAMBA_SLOTS:-18}
      --min-free-slots-delay 1 --enable-linear-replayssm-spec --disable-prefill-cuda-graph --linear-attn-verify-backend triton
      --tool-call-parser qwen3_coder --image-processor-backend pil ${ARC_SESSIONS:+--enable-streaming-session}
      --speculative-algorithm DFLASH --speculative-num-draft-tokens 4 --speculative-draft-model-quantization unquant
      --weight-loader-prefetch-checkpoints --model-loader-extra-config '{"num_threads": 2}')
printf '%s\n' "${ARGS[@]}" > server_args.txt   # the seed check compares the seed's command line with this one
CHECK=${JIT_SEED_CHECK:-$PWD/jit_seed_check.py}   # written by its own cell, beside this file
if [ "${1:-}" = pack ]; then   # the seed builder: this container's caches, after a warm boot and a real boot
    exec /tmp/sgl/bin/python "$CHECK" pack --out "$2" --args-file server_args.txt --fingerprint /tmp/jit_seed_fingerprint.json
fi
if [ "${1:-}" = check ]; then  # the seed check alone, on a seed already unpacked into /tmp/cache
    exec /tmp/sgl/bin/python "$CHECK" verify --seed "$2" --args-file server_args.txt
fi
stop_all() {   # the launcher, its TP workers and detokenizer, and any Python of ours still holding a GPU; bounded waits
    pkill -f "[s]glang.launch_server"; sleep 5; pkill -9 -f "[s]glang.launch_server"; pkill -9 -f "[s]glang::"
    # Only PIDs that are ours: in a gVisor sandbox nvidia-smi reports PID 1, the container's init (killing it ended a
    # Modal box twice, 1024_1311 / 1024_1333); on Kaggle it reports the real PIDs, so nothing changes there.
    for pid in $(timeout 20 nvidia-smi --query-compute-apps=pid --format=csv,noheader 2>/dev/null); do
        [ "$pid" -gt 1 ] 2>/dev/null && grep -qa python "/proc/$pid/cmdline" 2>/dev/null && kill -9 "$pid" 2>/dev/null
    done
    for i in $(seq 60); do [ -z "$(timeout 20 nvidia-smi --query-compute-apps=pid --format=csv,noheader 2>/dev/null)" ] && break; sleep 2; done
}
# The warm boot: compiles every JIT kernel with the plain NCCL all-reduce (the CPU reducer off), sends the warm-up
# requests, and stops; the real server then starts on a warm cache. It runs only when no seed covers this build.
warm_boot() {
    T0=$(date +%s)
    echo "warm boot: NCCL all-reduce, compiling the JIT kernels into /tmp/cache ($(date +%H:%M:%S))"
    env -u SGLANG_ENABLE_SM_REDUCE_ALLREDUCE -u SGLANG_CRASH_ON_JIT_COMPILE nohup setsid /tmp/sgl/bin/python -m sglang.launch_server "${ARGS[@]}" \
        > sglang_warm.log 2>&1 < /dev/null &
    WPID=$!
    for i in $(seq 480); do
        curl -sf -m 5 http://127.0.0.1:30000/health > /dev/null && break
        kill -0 $WPID 2> /dev/null || { echo "warm boot exited: $(tail -3 sglang_warm.log)"; break; }
        [ $((i % 12)) -eq 0 ] && echo "  warm boot +$(( $(date +%s) - T0 )) s: $(grep -c . sglang_warm.log) log lines"
        sleep 5
    done
    /tmp/sgl/bin/python warm_requests.py || echo "warm requests failed; the real server starts anyway"
    stop_all
    touch /tmp/cache/warm_done
    echo "warm boot done in $(( $(date +%s) - T0 )) s; cache $(du -sh /tmp/cache | cut -f1)"
}
start_server() {
    nohup setsid /tmp/sgl/bin/python -m sglang.launch_server "${ARGS[@]}" > sglang.log 2>&1 < /dev/null &
    SPID=$!
    echo "server starting (pid $SPID)"
}
# The seed: the compile cache of an earlier run of this exact build (an attached input). Used only when the check
# finds every seeded file intact and everything the cache keys depend on unchanged; otherwise the cache is emptied
# and the warm boot runs as before. A seeded boot that still dies before /health also falls back to the warm boot.
SEEDED=0
if [ ! -f /tmp/cache/warm_done ]; then
    SEED=${JIT_SEED_DIR:-$(dirname "$(find -L /kaggle/input -maxdepth 5 -name jit_seed_manifest.json 2>/dev/null | head -1)")}
    if [ -f "$SEED/jit_seed.tar" ] && [ -f "$CHECK" ]; then
        TS=$(date +%s)
        tar -C / -xf "$SEED/jit_seed.tar"   # the builder's mtimes kept: ninja reads them (the next line)
        # sm_reduce's extension is rebuilt by ninja whenever an input looks newer than its objects, and this install's
        # sources and headers all do: date them back (ninja never reads contents; the check below compares those).
        /tmp/sgl/bin/python "$CHECK" torchext | tee seed_torchext.json
        if /tmp/sgl/bin/python "$CHECK" verify --seed "$SEED" --args-file server_args.txt | tee seed_check.json | grep -q '"ok": true'; then
            SEEDED=1
            touch /tmp/cache/warm_done
            echo "seeded compile cache from $SEED: $(du -sh /tmp/cache | cut -f1) in $(( $(date +%s) - TS )) s; one boot"
        else
            echo "seed does not cover this build ($(cut -c1-400 seed_check.json)); emptying the cache for the warm boot"
            rm -rf /tmp/cache/* && mkdir -p /tmp/cache/{triton,inductor,flashinfer,xdg,torch_extensions,sglang_jit,sglang,cute_dsl}
        fi
    else
        echo "no compile-cache seed attached"
    fi
    if [ $SEEDED = 0 ]; then
        # What the caches' keys depend on, taken before any boot: lets this container's caches be packed as a seed
        # later (`launch_server.sh pack DIR`), checked in the same state a reusing container is checked in.
        [ -f "$CHECK" ] && /tmp/sgl/bin/python "$CHECK" fingerprint --out /tmp/jit_seed_fingerprint.json --args-file server_args.txt
        warm_boot
    fi
fi
touch /tmp/jit_seed_boot_marker   # a cache hit writes no file: anything newer than this under /tmp/cache was compiled
start_server
if [ $SEEDED = 1 ]; then   # wait here, so a boot that dies can still fall back
    TB=$(date +%s)
    for i in $(seq 720); do
        curl -sf -m 5 http://127.0.0.1:30000/health > /dev/null && break
        kill -0 $SPID 2> /dev/null || break
        [ $((i % 12)) -eq 0 ] && echo "  seeded boot +$(( $(date +%s) - TB )) s: $(grep -c . sglang.log) log lines"
        sleep 5
    done
    if ! curl -sf -m 5 http://127.0.0.1:30000/health > /dev/null; then
        echo "seeded boot did not come up ($(tail -2 sglang.log | cut -c1-300)); falling back to the warm boot"
        stop_all
        mv sglang.log "sglang_seeded_failed_$(date +%m%d_%H%M%S).log"
        rm -rf /tmp/cache/* && mkdir -p /tmp/cache/{triton,inductor,flashinfer,xdg,torch_extensions,sglang_jit,sglang,cute_dsl}
        /tmp/sgl/bin/python "$CHECK" fingerprint --out /tmp/jit_seed_fingerprint.json --args-file server_args.txt
        unset SGLANG_CRASH_ON_JIT_COMPILE   # (a test instrument only) the fallback compiles what it needs
        warm_boot
        start_server
    else
        echo "seeded boot up in $(( $(date +%s) - TB )) s; new files in the compile cache (misses):" \
             "$(find /tmp/cache -type f -newer /tmp/jit_seed_boot_marker -not -path '*/tilelang/tmp/*' -not -path '*/.staging*' | wc -l);" \
             "target-verify capture $(grep -o 'Capture target verify CUDA graph end. elapsed=[0-9.]*' sglang.log | head -1 | cut -d= -f2) s"
    fi
fi


In [ ]:
%%writefile warm_requests.py
# The warm boot's requests: a two-chunk prefill with a decode, and a picture, so the prefill, decode and vision
# kernels are compiled before the real server starts. Standard library and Pillow only.
import base64, io, json, time, urllib.request
from PIL import Image

def post(path, body, timeout=900):
    req = urllib.request.Request('http://127.0.0.1:30000' + path, data=json.dumps(body).encode(),
                                 headers={'Content-Type': 'application/json'})
    return json.loads(urllib.request.urlopen(req, timeout=timeout).read())

grid = ' '.join(str((i * 7) % 10) for i in range(900))   # ~1,800 tokens: two 1,024-token prefill chunks
t = time.time()
r = post('/v1/chat/completions', {'model': 'qwen3.8-flash-next', 'max_tokens': 64, 'temperature': 1.0,
                                  'messages': [{'role': 'user', 'content': 'Describe this grid briefly: ' + grid}]})
print(f'warm text request: {r["usage"]["completion_tokens"]} tokens in {time.time() - t:.0f} s', flush=True)
img = Image.new('RGB', (240, 240), (0, 0, 0))
for x in range(0, 240, 24):
    for y in range(0, 240, 24):
        img.paste(((x * 3) % 255, (y * 5) % 255, 128), (x, y, x + 20, y + 20))
buf = io.BytesIO()
img.save(buf, format='PNG')
url = 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode()
t = time.time()
r = post('/v1/chat/completions', {'model': 'qwen3.8-flash-next', 'max_tokens': 32, 'temperature': 1.0,
                                  'messages': [{'role': 'user', 'content': [
                                      {'type': 'image_url', 'image_url': {'url': url}},
                                      {'type': 'text', 'text': 'How many squares are there?'}]}]})
print(f'warm picture request: {r["usage"]["completion_tokens"]} tokens in {time.time() - t:.0f} s', flush=True)


**The seed check**, as a file the launch runs: `fingerprint` before a cold container's first boot, `pack` to write a
seed, `verify` to check one before it is used, and `torchext`, which dates the CPU all-reduce extension's sources back
so its build tool sees the seeded build as current (it compares file times, and this container's install is newer).
Standard library plus the fork's own cache code.


In [ ]:
%%writefile jit_seed_check.py
"""The seeded compile cache: pack it after a warm boot, and check it before a boot that relies on it.

JIT-cache seeding (docs/reports/1024_1137_jit_cache_seeding/): the Kaggle notebook boots its server once with every
kernel already compiled, from a cache built by an earlier run of the same build. A cache that does not cover this
exact build would compile at boot instead, and a long compile on one GPU while the others spin in sm_reduce's
all-reduce trips its 50M-poll trap (sm_reduce_kernel.cu:406-408, 497-502). So the boot trusts the seed only when this
check passes, and otherwise falls back to today's warm boot on an EMPTY cache.

    python jit_seed_check.py fingerprint --out FILE --args-file server_args.txt   (first launch, before any boot)
    python jit_seed_check.py pack   --out DIR --args-file server_args.txt [--fingerprint FILE] [--dir PATH ...]
    python jit_seed_check.py verify --seed DIR --args-file server_args.txt [--root /]

Run it with the server's own interpreter (/tmp/sgl/bin/python) and environment (inside launch_server.sh, after the
exports): it imports the fork's own JIT-cache code for two checks, so they are exactly the server's.

`pack` writes DIR/jit_seed.tar (paths relative to /) and DIR/jit_seed_manifest.json:
- every file with its size and sha256 (lock files and in-progress build directories left out);
- the fingerprint of everything the caches' keys depend on (below);
- a summary per cache.

`verify` (after `tar -C / -xmf DIR/jit_seed.tar`) exits 0 when the seed covers this build, else 3, and prints one
JSON line. It fails on any HARD field that differs, any seeded file missing or changed, or any SGLang JIT build whose
recorded dependencies (sources, CUDA and system headers) no longer hash to its own name. SOFT fields (driver, CPU
model, host) are reported only: no cache we seed is keyed on them, except the CUDA driver's own PTX cache, which a
different driver merely refills.

HARD fingerprint fields:
- gpu: the GPUs' names and compute capabilities (Triton, inductor, TileLang, SGLang JIT all key on the arch);
- cxx: `c++ --version` (the SGLang JIT key, sglang/kernels/jit/utils/compile/cache.py:241-245; host code of every
  nvcc build, and Triton's launcher stubs);
- nvcc: `$CUDA_HOME/bin/nvcc --version`;
- python: the interpreter's version;
- venv: every installed distribution's RECORD (file list with sha256), so any package difference shows;
- sglang_tree: every file of the installed, patched sglang package (the diff, the JIT sources, sm_reduce's sources);
- env: the server switches and cache paths the launch exports, every cache path by its effective value (CPU pinning
  lists and the allocator's loader-only `pinned_max_round_threshold_mb` left out);
- server_args: the server's command line, without the loader-only switches (LOADER_ONLY_FLAGS);
- sgl_jit_env: the fork's own `_environment_fingerprint()` (target, both compilers, package versions);
- cpu_isa: sm_reduce's ISA tag, part of its extension's name (a different ISA builds that extension anew: 33 s at
  rank 0 before any all-reduce runs, so it is SOFT for safety but reported).
"""

from __future__ import annotations

import argparse
import glob
import hashlib
import json
import os
import platform
import re
import subprocess
import sys
import tarfile
import time
from pathlib import Path

SEED_TAR = "jit_seed.tar"
MANIFEST = "jit_seed_manifest.json"
# 2 (2026-10-24): the cache paths SGLang's import sets are hashed by their effective value, and the loader-only
# switches below are left out. A format-1 seed is refused (a fallback, never a wrong hit).
FORMAT = 2
DEFAULT_DIRS = ["/tmp/cache"]
# Server switches that steer only how weights are read or host memory is pinned: no compiled artifact depends on them
# (cache-key study docs/reports/1024_1137_jit_cache_seeding/), so tuning them must not invalidate the seed.
LOADER_ONLY_FLAGS = {"--weight-loader-prefetch-checkpoints": 0, "--model-loader-extra-config": 1}   # flag -> values
LOADER_ONLY_ALLOC = ("pinned_max_round_threshold_mb",)   # parts of PYTORCH_CUDA_ALLOC_CONF
# What `import sglang` sets with os.environ.setdefault (fork environ.py third_party_cache_defaults): hashed by the value
# in force either way, so a caller that imported sglang (and so set them) fingerprints the same as one that did not.
SGLANG_CACHE_DEFAULTS = {"TRITON_CACHE_DIR": "triton", "TORCHINDUCTOR_CACHE_DIR": "inductor", "CUDA_CACHE_PATH": "nv",
                         "TILELANG_CACHE_DIR": "tilelang", "FLASHINFER_WORKSPACE_BASE": ""}
# Never seeded: the warm-boot marker (the boot decides that), torch's build baton (a stale `lock` makes
# cpp_extension.load wait for a builder that does not exist), SGLang JIT staging dirs (builds in progress).
SKIP_NAMES = {"warm_done", "lock"}
SKIP_PREFIXES = (".staging-",)
ENV_KEYS_EXTRA = ("CUDA_HOME", "NVCC_APPEND_FLAGS", "TRITON_CACHE_DIR", "TORCHINDUCTOR_CACHE_DIR",
                  "SGLANG_JIT_CACHE_DIR", "FLASHINFER_WORKSPACE_BASE", "XDG_CACHE_HOME", "TORCH_EXTENSIONS_DIR",
                  "SGLANG_CACHE_DIR", "TILELANG_CACHE_DIR", "CUDA_CACHE_PATH", "CUTE_DSL_CACHE_DIR", "PYTORCH_CUDA_ALLOC_CONF",
                  "ARC_SLOTS", "ARC_MAX_REQS", "ARC_SESSIONS", "HOME")
# Families whose variables enter cache keys (Triton's invalidating set, inductor's config, FLA's autotune cache,
# CuTe DSL, FlashInfer, TileLang) or the rank -> GPU mapping inductor keys carry (CUDA_VISIBLE_DEVICES).
ENV_PREFIXES = ("SGLANG_", "TRITON_", "TORCHINDUCTOR_", "TORCH_COMPILE", "TORCHDYNAMO", "FLA_", "PTXAS", "MLIR_",
                "LLVM_", "NVPTX_", "CUTE_DSL", "FLASHINFER_", "TILELANG_", "CUDA_VISIBLE_DEVICES")
# Switches that change no kernel: CPU pinning (differs per machine), the check's own diagnostics, and the PLE direct
# read's loader switches (fork agi2/feat/ple-direct: host-side preadv into the pinned tables, which hold the same bytes,
# and which files the prefetch skips; sub 13, 2026-10-25), so the PLE fix can be switched without a new seed.
ENV_SKIP = {"SGLANG_SM_REDUCE_CPUS", "SGLANG_CRASH_ON_JIT_COMPILE", "SGLANG_JIT_CACHE_DEBUG",
            "SGLANG_TRITON_SLOW_COMPILE_THRESHOLD_SECS",
            "SGLANG_AGI2_PLE_DIRECT_READ", "SGLANG_AGI2_PLE_DIRECT_THREADS", "SGLANG_AGI2_PREFETCH_EXCLUDE",
            # per-token confidence (agi2/feat/fn-conf): with it on, boot and graph capture are unchanged; its two Triton
            # kernels compile only at the first request that asks (the logprobs session, 2026-10-25). It is safe with
            # SGLANG_ENABLE_GRAPH_POOL_BORROW: no borrowed scope reaches its q stash (the earlier hazard was a misread;
            # the logprobs session, 10-25 ~10:00 IST). The borrow switch stays in the fingerprint for its own sake.
            "SGLANG_AGI2_FN_CONF", "SGLANG_AGI2_FN_CONF_CHECK"}
HARD = ("gpu", "cxx", "nvcc", "python", "venv", "sglang_tree", "env", "server_args", "sgl_jit_env")
SOFT = ("driver", "cpu_isa", "cpu_model", "glibc", "host")


def _sha(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def _file_sha(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def _run(cmd: list[str]) -> str:
    try:
        return subprocess.run(cmd, capture_output=True, text=True, timeout=60).stdout.strip()
    except (OSError, subprocess.SubprocessError) as e:
        return f"ERROR {type(e).__name__}: {e}"


def _site() -> Path:
    import sysconfig

    return Path(sysconfig.get_paths()["purelib"])


def _venv_hash() -> str:
    lines = []
    for rec in sorted(glob.glob(str(_site() / "*.dist-info" / "RECORD"))):
        for line in open(rec, encoding="utf-8", errors="replace"):
            path = line.split(",", 1)[0]
            if "__pycache__" in path or path.endswith(("INSTALLER", "REQUESTED", "direct_url.json", "uv_cache.json")):
                continue
            lines.append(line.rstrip("\n"))
    return _sha("\n".join(lines).encode())


def _tree_hash(root: Path) -> tuple[str, int]:
    h, n = hashlib.sha256(), 0
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = sorted(d for d in dirnames if d != "__pycache__")
        for name in sorted(filenames):
            if name.endswith(".pyc"):
                continue
            p = os.path.join(dirpath, name)
            h.update(os.path.relpath(p, root).encode() + b"\0" + _file_sha(p).encode() + b"\n")
            n += 1
    return h.hexdigest(), n


def _cpu_isa() -> str:
    """sm_reduce's own rule (sm_reduce_allreduce.py `_cpu_isa`): the tag in its extension's name."""
    try:
        flags = set(Path("/proc/cpuinfo").read_text().split("flags", 2)[1].split("\n", 1)[0].split())
    except (OSError, IndexError):
        flags = set()
    if {"avx512f", "avx512bw", "avx512vl"} <= flags:
        return "avx512"
    if {"avx2", "fma"} <= flags:
        return "avx2"
    return "generic"


def _cpu_model() -> str:
    for line in Path("/proc/cpuinfo").read_text().splitlines():
        if line.startswith("model name"):
            return line.split(":", 1)[1].strip()
    return "?"


def _effective_env() -> dict[str, str]:
    env = {k: v for k, v in os.environ.items()
           if (k.startswith(ENV_PREFIXES) or k in ENV_KEYS_EXTRA) and k not in ENV_SKIP}
    base = os.path.expanduser(os.environ.get("SGLANG_CACHE_DIR", "~/.cache/sglang"))
    for k, sub in SGLANG_CACHE_DEFAULTS.items():
        env.setdefault(k, os.path.join(base, sub) if sub else base)
    if "PYTORCH_CUDA_ALLOC_CONF" in env:
        parts = [p for p in env["PYTORCH_CUDA_ALLOC_CONF"].split(",") if p.split(":")[0] not in LOADER_ONLY_ALLOC]
        env["PYTORCH_CUDA_ALLOC_CONF"] = ",".join(parts)
    return env


def _env_hash() -> tuple[str, list[str]]:
    env = _effective_env()
    items = [f"{k}={env[k]}" for k in sorted(env)]
    return _sha("\n".join(items).encode()), items


def _args_hash(args_file: str) -> str:
    """The server's command line (one argument a line), loader-only switches and their values left out."""
    args, out, skip = open(args_file, encoding="utf-8").read().splitlines(), [], 0
    for a in args:
        if skip:
            skip -= 1
        elif a in LOADER_ONLY_FLAGS:
            skip = LOADER_ONLY_FLAGS[a]
        else:
            out.append(a)
    return _sha("\n".join(out).encode())


def _sgl_jit_env() -> str:
    from sglang.kernels.jit.utils.compile import cache  # the fork's own key code

    return cache._environment_fingerprint()


def fingerprint(args_file: str) -> tuple[dict, dict]:
    """(fields, detail): the fields compared, and the readable detail behind the hashed ones."""
    t = time.time()
    smi = _run(["nvidia-smi", "--query-gpu=name,compute_cap,driver_version", "--format=csv,noheader"]).splitlines()
    cuda_home = os.environ.get("CUDA_HOME", "")
    env_h, env_items = _env_hash()
    tree_h, tree_n = _tree_hash(_site() / "sglang")
    fields = {
        "gpu": sorted({",".join(x.strip() for x in row.split(",")[:2]) for row in smi}) + [f"count={len(smi)}"],
        "cxx": _run(["c++", "--version"]),
        "nvcc": _run([os.path.join(cuda_home, "bin", "nvcc"), "--version"]).splitlines()[-2:],
        "python": sys.version,
        "venv": _venv_hash(),
        "sglang_tree": tree_h,
        "env": env_h,
        "server_args": _args_hash(args_file),
        "sgl_jit_env": _sgl_jit_env(),
        "driver": sorted({row.split(",")[2].strip() for row in smi if row.count(",") >= 2}),
        "cpu_isa": _cpu_isa(),
        "cpu_model": _cpu_model(),
        "glibc": _run(["ldd", "--version"]).splitlines()[:1],
        "host": platform.node(),
    }
    detail = {"env": env_items, "sglang_files": tree_n, "fingerprint_s": round(time.time() - t, 2)}
    return fields, detail


def _seed_files(dirs: list[str]) -> list[str]:
    out = []
    for d in dirs:
        for dirpath, dirnames, filenames in os.walk(d):
            dirnames[:] = sorted(x for x in dirnames if not x.startswith(SKIP_PREFIXES))
            for name in sorted(filenames):
                p = os.path.join(dirpath, name)
                if name in SKIP_NAMES or os.path.islink(p) or not os.path.isfile(p):
                    continue
                out.append(p)
    return out


def _jit_leaves(root: str) -> list[Path]:
    jit = Path(root + os.environ.get("SGLANG_JIT_CACHE_DIR", "/tmp/cache/sglang_jit"))
    return sorted(p.parent for p in jit.glob("*/*/build-*/deps-*/sgl_deps.json"))


def _check_leaf(leaf: Path) -> str | None:
    """None if the leaf is a hit as the server would see it (cache.find_prebuilt's test), else why not."""
    from sglang.kernels.jit.utils.compile import cache

    entries = cache._read_deps(leaf)
    if entries is None:
        return "unreadable sgl_deps.json"
    current, changed = cache._refresh(entries)
    if current is None:
        return f"dependency gone: {changed[0]}"
    if f"deps-{cache._deps_key(current)}" != leaf.name:
        return f"dependencies changed: {', '.join(changed[:3])}"
    module = leaf.parent.parent.name
    if not (leaf / f"{module}.so").is_file():
        return f"{module}.so missing"
    return None


def _summary(files: list[str]) -> dict:
    def size(prefix):
        return sum(os.path.getsize(f) for f in files if f.startswith(prefix))

    jit = os.environ.get("SGLANG_JIT_CACHE_DIR", "/tmp/cache/sglang_jit")
    return {
        "files": len(files), "bytes": sum(os.path.getsize(f) for f in files),
        "sglang_jit_modules": sorted({Path(f).relative_to(jit).parts[1] for f in files
                                      if f.startswith(jit + "/") and f.endswith(".so")}),
        "sglang_jit_leaves": len([f for f in files if f.startswith(jit + "/") and f.endswith("sgl_deps.json")]),
        "torch_extensions": sorted({Path(f).relative_to(os.environ.get("TORCH_EXTENSIONS_DIR", "/tmp/cache/torch_extensions")).parts[0]
                                    for f in files if f.startswith(os.environ.get("TORCH_EXTENSIONS_DIR", "/tmp/cache/torch_extensions") + "/")}),
        "bytes_by_dir": {d: size(d) for d in sorted({"/".join(f.split("/")[:4]) for f in files})},
    }


def _ninja_inputs(build_dir: str) -> tuple[list[str], list[str]]:
    """(inputs, outputs) of a torch extension's ninja build: explicit inputs from build.ninja plus every header its
    deps log lists (`ninja -t deps`), and the outputs (objects, the library)."""
    ins, outs = set(), set()
    text = open(os.path.join(build_dir, "build.ninja"), encoding="utf-8").read().replace("$\n", " ")
    for line in text.splitlines():
        if line.startswith("build "):
            # the outputs end at the first unescaped ':' ("$:" is ninja's escaped colon)
            lhs, rhs = re.split(r"(?<!\$):", line[len("build "):], maxsplit=1)
            unesc = lambda t: t.replace("$:", ":").replace("$$", "$")  # noqa: E731
            outs.update(os.path.join(build_dir, unesc(o)) for o in lhs.split())
            for tok in rhs.split()[1:]:   # after the rule name; skip the implicit / order-only markers
                if tok not in ("|", "||") and os.path.isabs(unesc(tok)):
                    ins.add(unesc(tok))
    deps = subprocess.run(["ninja", "-C", build_dir, "-t", "deps"], capture_output=True, text=True).stdout
    for ln in deps.splitlines():   # a target's line, then its dependencies indented (relative ones to the build dir)
        p = ln.strip()
        if ln.startswith((" ", "\t")) and p:
            ins.add(p if os.path.isabs(p) else os.path.normpath(os.path.join(build_dir, p)))
    return sorted(ins), sorted(o for o in outs if os.path.exists(o))


def torchext(a) -> int:
    """Make a seeded torch extension build directory current for ninja, then ask ninja (dry run) whether it would build.

    torch's cpp_extension.load runs ninja on every load, and ninja rebuilds when any input (the sources, every header in
    its deps log) is newer than the objects: a fresh install re-copies all of them with new mtimes. Extracting with
    `tar -m` does not help (the deps log then reads "stored deps info out of date"). So the seed keeps the builder's
    mtimes, and here every input newer than an hour before the oldest object is dated back to that time (ninja never
    hashes content; what the inputs contain is the fingerprint's job: venv, sglang_tree, cxx, nvcc). A stale `lock`
    would make load() wait forever, so it is removed."""
    root = os.environ.get("TORCH_EXTENSIONS_DIR", "/tmp/cache/torch_extensions")
    out = []
    for d in sorted(glob.glob(os.path.join(root, "*", "build.ninja"))):
        bd = os.path.dirname(d)
        lock = os.path.join(bd, "lock")
        if os.path.exists(lock):
            os.remove(lock)
        ins, outs = _ninja_inputs(bd)
        if not outs:
            out.append({"dir": bd, "ninja": "no outputs"})
            continue
        t_ref = min(os.path.getmtime(o) for o in outs) - 3600
        moved = 0
        for p in ins:
            if os.path.exists(p) and os.path.getmtime(p) > t_ref:
                os.utime(p, (t_ref, t_ref))
                moved += 1
        dry = subprocess.run(["ninja", "-C", bd, "-n", "-d", "explain"], capture_output=True, text=True)
        said = (dry.stdout + dry.stderr).strip().splitlines()
        current = any("no work to do" in ln for ln in said)
        out.append({"dir": os.path.basename(bd), "inputs": len(ins), "dated_back": moved, "current": current,
                    "ninja": said[-3:] if not current else "no work to do"})
    ok = all(r.get("current") for r in out)
    print(json.dumps({"torch_extensions": out, "ok": ok}), flush=True)
    return 0 if ok else 3


def save_fingerprint(a) -> int:
    """Taken by the first launch in a container before any boot, so a later `pack` records the state a reusing
    container is checked in (fresh install and patch, nothing run yet)."""
    fields, detail = fingerprint(a.args_file)
    with open(a.out, "w", encoding="utf-8") as f:
        json.dump({"fingerprint": fields, "detail": detail}, f, indent=1)
    print(json.dumps({"fingerprint": a.out, "s": detail["fingerprint_s"]}), flush=True)
    return 0


def pack(a) -> int:
    t0 = time.time()
    dirs = a.dir or DEFAULT_DIRS
    files = _seed_files(dirs)
    os.makedirs(a.out, exist_ok=True)
    if a.fingerprint and os.path.exists(a.fingerprint):
        saved = json.load(open(a.fingerprint, encoding="utf-8"))
        fields, detail = saved["fingerprint"], saved["detail"]
        now, _ = fingerprint(a.args_file)
        drift = [k for k in HARD if now.get(k) != fields.get(k)]
        if drift:   # the boots changed what the keys depend on: a reusing container would never match
            print(json.dumps({"pack": "refused", "changed_since_first_launch": drift}), flush=True)
            return 3
    else:
        fields, detail = fingerprint(a.args_file)
    leaves = _jit_leaves("")
    bad = {str(leaf): why for leaf in leaves if (why := _check_leaf(leaf))}
    if bad:
        print(json.dumps({"pack": "refused", "bad_leaves": bad}), flush=True)
        return 3
    entries = [{"path": f, "size": os.path.getsize(f), "sha256": _file_sha(f)} for f in files]
    # The torch extension as ninja sees it on the builder: what its objects depend on, and that it is current here.
    ext = []
    for d in sorted(glob.glob(os.path.join(os.environ.get("TORCH_EXTENSIONS_DIR", "/tmp/cache/torch_extensions"),
                                           "*", "build.ninja"))):
        bd = os.path.dirname(d)
        ins, outs = _ninja_inputs(bd)
        dry = subprocess.run(["ninja", "-C", bd, "-n"], capture_output=True, text=True)
        ext.append({"dir": os.path.basename(bd), "inputs": len(ins), "outputs": [os.path.basename(o) for o in outs],
                    "current_on_builder": "no work to do" in dry.stdout + dry.stderr})
    # GNU format: whole-second mtimes, rounded down. Ninja needs each restored object no newer than its deps-log record,
    # which a sub-second mtime rounded up (PAX floats) could break.
    with tarfile.open(os.path.join(a.out, SEED_TAR), "w", format=tarfile.GNU_FORMAT) as tar:
        for f in files:
            tar.add(f, arcname=f.lstrip("/"), recursive=False)
    manifest = {"format": FORMAT, "built": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
                "dirs": dirs, "hard": HARD, "soft": SOFT, "fingerprint": fields, "detail": detail,
                "summary": {**_summary(files), "torch_extensions_ninja": ext}, "files": entries}
    with open(os.path.join(a.out, MANIFEST), "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=1)
    s = manifest["summary"]
    print(json.dumps({"pack": "ok", "files": s["files"], "MB": round(s["bytes"] / 2**20, 1),
                      "sglang_jit_modules": s["sglang_jit_modules"], "torch_extensions": s["torch_extensions"],
                      "leaves": len(leaves), "s": round(time.time() - t0, 1)}), flush=True)
    return 0


def verify(a) -> int:
    t0 = time.time()
    try:
        manifest = json.load(open(os.path.join(a.seed, MANIFEST), encoding="utf-8"))
    except (OSError, ValueError) as e:
        print(json.dumps({"ok": False, "why": [f"no manifest: {type(e).__name__}: {e}"]}), flush=True)
        return 3
    why, soft = [], {}
    if manifest.get("format") != FORMAT:
        why.append(f"manifest format {manifest.get('format')} != {FORMAT}")
    root = a.root.rstrip("/")
    missing = changed = 0
    for e in manifest["files"]:
        p = root + e["path"]
        if not os.path.isfile(p):
            missing += 1
        elif os.path.getsize(p) != e["size"] or _file_sha(p) != e["sha256"]:
            changed += 1
    if missing or changed:
        why.append(f"seeded files: {missing} missing, {changed} changed of {len(manifest['files'])}")
    fields, detail = fingerprint(a.args_file)
    seed_fp = manifest["fingerprint"]
    for k in HARD:
        if fields.get(k) != seed_fp.get(k):
            why.append(f"{k} differs: seed {str(seed_fp.get(k))[:120]!r} vs here {str(fields.get(k))[:120]!r}")
    for k in SOFT:
        if fields.get(k) != seed_fp.get(k):
            soft[k] = {"seed": seed_fp.get(k), "here": fields.get(k)}
    leaves = _jit_leaves(root)
    want = manifest["summary"]["sglang_jit_leaves"]
    if len(leaves) != want:
        why.append(f"SGLang JIT leaves: {len(leaves)} here vs {want} seeded")
    bad = {str(leaf): w for leaf in leaves if (w := _check_leaf(leaf))}
    if bad:
        why.append(f"SGLang JIT builds that would rebuild: {bad}")
    if "env" in [w.split(" ")[0] for w in why]:
        seed_env = set(manifest["detail"].get("env", []))
        here_env = set(detail["env"])
        why.append(f"env only in seed: {sorted(seed_env - here_env)}; only here: {sorted(here_env - seed_env)}")
    verdict = {"ok": not why, "why": why, "soft_diff": soft, "files": len(manifest["files"]),
               "leaves": len(leaves), "seed_built": manifest.get("built"), "s": round(time.time() - t0, 1)}
    print(json.dumps(verdict), flush=True)
    return 0 if not why else 3


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__.split("\n\n")[0])
    sub = ap.add_subparsers(dest="cmd", required=True)
    p = sub.add_parser("fingerprint")
    p.add_argument("--out", required=True)
    p.add_argument("--args-file", required=True)
    p = sub.add_parser("pack")
    p.add_argument("--out", required=True)
    p.add_argument("--args-file", required=True)
    p.add_argument("--fingerprint", help="the fingerprint the first launch saved (default: take one now)")
    p.add_argument("--dir", action="append", help="a cache directory to seed (default /tmp/cache); repeatable")
    p = sub.add_parser("verify")
    p.add_argument("--seed", required=True, help="the directory holding jit_seed_manifest.json")
    p.add_argument("--args-file", required=True)
    p.add_argument("--root", default="/", help="where the seed was unpacked (default /)")
    sub.add_parser("torchext", help="date a seeded torch extension's inputs back so ninja finds it current")
    a = ap.parse_args()
    return {"fingerprint": save_fingerprint, "pack": pack, "verify": verify, "torchext": torchext}[a.cmd](a)


if __name__ == "__main__":
    sys.exit(main())


**Start the server**, with host RAM logged from here on:


In [ ]:
%%bash
# Host RAM every 10 s from here on (used and available MiB): the pinned tables and the page cache during the boot
nohup setsid bash -c 'while true; do echo "$(date +%H:%M:%S) $(free -m | grep Mem:)"; sleep 10; done' > host_mem.log 2>&1 < /dev/null &
echo $! > host_mem.pid
free -g | head -2


In [ ]:
%%bash
# %%bash, not `!`: `!` runs on a terminal that hangs up when the script ends, which killed the server before it started (17:28)
bash launch_server.sh

In [ ]:
import re, subprocess, requests

SERVER = 'http://127.0.0.1:30000'
log_pos = 0

def show_server_log():
    """Print the server log written since the last call; GPUs 1-3 repeat GPU 0's lines, so only GPU 0's are shown."""
    global log_pos
    with open('sglang.log', 'rb') as f:
        f.seek(log_pos)
        text = f.read().decode(errors='replace')
        log_pos = f.tell()
    for line in text.splitlines():
        if not re.search(r' TP[1-3][ \]]', line):
            print(line, flush=True)

t = time.time()
SERVER_UP = False
while time.time() - t < 3600:   # 60 min: /kaggle/input has read ~24 MB/s at times since 10-20
    show_server_log()
    try:
        SERVER_UP = requests.get(f'{SERVER}/health', timeout=5).status_code == 200
    except requests.RequestException:
        pass
    if SERVER_UP or subprocess.run(['pgrep', '-f', 'sglang.launch_server'], capture_output=True).returncode:
        break
    time.sleep(5)
show_server_log()
check(SERVER_UP, 'the server did not come up (its log is above)')
print(f'server up: {SERVER_UP} ({time.time() - t:.0f} s in this cell)')

In [ ]:
# The boot facts, from the server log: a failed one stops the run
if SERVER_UP:
    boot = open('sglang.log', errors='replace').read()
    pool = re.search(r'TP0[^\]]*\] KV Cache is allocated.*?#tokens: (\d+)', boot)
    keep = re.search(r'agi2 expert keep: (\d+) experts per layer kept', boot)
    draft_graph = ('disabled' if 'Disable DFLASH draft cuda graph' in boot else
                   'captured' if 'Capture draft verify CUDA graph end' in boot else 'unknown')
    free = re.findall(r'TP\d[^\]]*\] Capture draft verify CUDA graph begin.*?avail mem=([\d.]+) GB', boot)
    print(f"KV pool {int(pool[1]) if pool else 0:,} tokens a GPU; experts kept {keep[1] if keep else '?'}; "
          f"draft CUDA graph {draft_graph}; free before the draft graphs {free} GB")
    check(keep is not None and keep[1] == '384', 'the 384-expert keep list did not load')
    check(draft_graph == 'captured', f'the DFlash draft CUDA graph is {draft_graph} (needs >= 1.0 GB free)')
    check(pool is not None and int(pool[1]) > 600_000, 'the KV pool is smaller than the build gives (~4.7M with the block tier at 7.5x)')
    tier = re.search(r'TP0[^\]]*\] agi2 QSA block tier plan: .*?-> index slots (\d+) .*?device rows (\d+), host rows (\d+)', boot)
    print('QSA block tier: ' + (f'{int(tier[1]):,} index slots, {int(tier[2]):,} GPU rows, {int(tier[3]):,} host rows'
                                if tier else 'NOT ON'))
    check(tier is not None and 'OFF (host-NUMA pages unavailable' not in boot,
          'the QSA block tier did not switch on (host-NUMA pages refused?): 48 slots would park')


**Serving:** the same two requests as the warm boot's, against the real server.


In [ ]:
!/tmp/sgl/bin/python warm_requests.py
!free -g | head -2 && echo "peak used MiB: $(sort -k4 -n host_mem.log | tail -1)"
print(f'serving at {time.time() - T0:.0f} s since the settings cell')


## 6. The seed

The caches of this container, after both boots and the requests, with the fingerprint taken before the first
boot. `pack` refuses if anything the caches' keys depend on changed in between.


In [ ]:
%%bash
bash launch_server.sh pack /kaggle/working/jit_seed
ls -la /kaggle/working/jit_seed


## 7. Stop

The server and every process on the GPUs are killed, and the cell waits until the GPUs are free. Every command here is
bounded and allowed to fail.


In [ ]:
%%bash
# Every command here is bounded and allowed to fail: this cell's exit status must never be what stops the notebook
echo "compiled into /tmp/cache after the boot: $(find /tmp/cache -type f -newer /tmp/jit_seed_boot_marker -not -path '*/tilelang/tmp/*' -not -path '*/.staging*' 2>/dev/null | wc -l) files; Triton compiles after serving started: $(grep -c 'after serving started' sglang.log 2>/dev/null); TileLang compiles: $(grep -c 'TileLang begins to compile' sglang.log 2>/dev/null)"
pkill -9 -f "[s]glang.launch_server" || true
pkill -9 -f "[s]glang::" || true
# only our Python PIDs: in a gVisor sandbox (Modal) nvidia-smi reports PID 1, the container's init
for pid in $(timeout 20 nvidia-smi --query-compute-apps=pid --format=csv,noheader 2>/dev/null); do
    [ "$pid" -gt 1 ] 2>/dev/null && grep -qa python "/proc/$pid/cmdline" 2>/dev/null && kill -9 "$pid" 2>/dev/null || true
done
for i in $(seq 60); do [ -z "$(timeout 20 nvidia-smi --query-compute-apps=pid --format=csv,noheader 2>/dev/null)" ] && break; sleep 2; done
timeout 20 nvidia-smi --query-gpu=index,memory.used --format=csv,noheader || true
kill "$(cat host_mem.pid 2>/dev/null)" 2>/dev/null || true   # the host RAM log
true
